<a href="https://colab.research.google.com/github/watanabe3tipapa/GoogleColaboratory/blob/main/mcp/pwa_colab_connection_guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PWA から Google Colab への接続ガイドProgressive Web App から Google Colab の GPU / TPU 計算資源を活用するための接続方法を、実装コードとともに体系的に解説します。2026年9月版 | FastAPI / WebSocket / MCP | ngrok トンネリング

## 全体構成と前提PWA から Google Colab に接続する際の基本的なアーキテクチャは、Colab ランタイム上に Web サーバを立ち上げ、トンネリングサービスで公開 URL を取得し、PWA からそのエンドポイントにアクセスする形になります。```PWA (フロントエンド) ブラウザ / スマホアプリ      ↑↓ HTTP / WSGoogle Colab FastAPI / Flask / MCP Server      ↓   GPU / TPU 計算実行```**前提条件**- Colab ランタイムはセッション継続中のみ有効です。一定時間アイドル状態が続くと切断されるため、keep-alive 対策が必要です。- 無料プランの ngrok では動的 URL が発行されるため、PWA 側で URL を更新できる仕組みを検討してください。

## FastAPI + ngrok による REST API 公開最も一般的かつ実装がシンプルな方法です。Colab 上で FastAPI サーバーを起動し、`pyngrok` でインターネットに公開します。PWA からは通常の HTTP リクエストで GPU 推論などを実行できます。

### API Colab 側: FastAPI サーバー（推奨）

In [ ]:
# パッケージインストール!pip install fastapi uvicorn pyngrok nest-asynciofrom fastapi import FastAPIfrom fastapi.middleware.cors import CORSMiddlewareimport nest_asyncioimport uvicornfrom pyngrok import ngrok# ngrok 認証トークン設定ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")app = FastAPI()# CORS を許可（PWA からのブラウザアクセスに必須）app.add_middleware(    CORSMiddleware,    allow_origins=["*"],    allow_credentials=True,    allow_methods=["*"],    allow_headers=["*"],)@app.get("/predict")def predict(text: str):    """GPU を使った推論処理の例"""    # ここに実際の推論コードを記述    return {"result": f"Processed: {text}"}# 公開 URL の取得public_url = ngrok.connect(8000)print(f"Public URL: {public_url}")# サーバー起動nest_asyncio.apply()uvicorn.run(app, host="0.0.0.0", port=8000)

### JS PWA 側: クライアント呼び出し```javascriptconst COLAB_API_URL = "https://xxxx.ngrok-free.app";  // ngrok URLasync function callColab(text) {  const response = await fetch(    `${COLAB_API_URL}/predict?text=${encodeURIComponent(text)}`  );  const data = await response.json();  return data.result;}```**ポイント**- `CORSMiddleware` の設定がないと、PWA からのブラウザリクエストがブロックされます。- ngrok の無料プランではセッションごとに URL が変わるため、PWA 側で動的に URL を更新する仕組みが必要です。

## WebSocket による双方向リアルタイム通信推論の進捗状況を PWA にリアルタイムで表示したい場合や、ストリーミングレスポンスが必要な場合は WebSocket が有効です。Colab 上に WebSocket サーバーを立ち上げ、ngrok で公開します。

### WS Colab 側: WebSocket サーバー

In [ ]:
!pip install websockets pyngrok nest-asyncioimport asyncioimport websocketsfrom pyngrok import ngrokimport nest_asynciongrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")async def handler(websocket, path):    async for message in websocket:        # GPU 推論などの処理        result = f"Result for: {message}"        await websocket.send(result)# WebSocket サーバー起動start_server = websockets.serve(handler, "0.0.0.0", 8765)# ngrok で WebSocket ポートを公開public_url = ngrok.connect(8765, "http")print(f"WebSocket URL: {public_url}")nest_asyncio.apply()asyncio.get_event_loop().run_until_complete(start_server)

### JS PWA 側: WebSocket クライアント```javascriptconst ws = new WebSocket("wss://xxxx.ngrok-free.app");ws.onopen = () => {  console.log("Connected to Colab WebSocket");  ws.send("Hello from PWA");};ws.onmessage = (event) => {  console.log("Received from Colab:", event.data);  // 進捗表示や結果レンダリング};ws.onerror = (error) => console.error("WebSocket error:", error);```Colab 上で WebSocket プロキシを構築するより高度な手法として、`google.colab.kernel.comms` を利用した JavaScript 連携も存在します。

## Colab MCP Server を Web サーバーとして公開2026年3月に Google から公開された Colab MCP Server は、AI エージェントが Colab ノートブックをプログラム的に操作するための仕組みです。デフォルトでは `stdio` トランスポートを使用しますが、HTTP / SSE / Streamable HTTP トランスポートに切り替えることで、PWA などの Web クライアントから直接アクセス可能な Web サーバーとして機能させることができます。

### 方法 A: MCP + SSE over ngrokMCP Python SDK の `SseServerTransport` を FastAPI 上で動作させ、SSE エンドポイントを公開します。MCP プロトコルに準拠したツール呼び出しが可です。

#### MCP Colab 側: MCP over SSE サーバー（上級）

In [ ]:
!pip install mcp fastapi uvicorn pyngrok nest-asyncioimport nest_asyncioimport uvicornfrom fastapi import FastAPIfrom fastapi.middleware.cors import CORSMiddlewarefrom pyngrok import ngrokfrom mcp.server.fastmcp import FastMCPfrom mcp.server.sse import SseServerTransportfrom starlette.requests import Requestngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")app = FastAPI()app.add_middleware(    CORSMiddleware,    allow_origins=["*"],    allow_methods=["*"],    allow_headers=["*"],)# MCP サーバー作成mcp = FastMCP("colab-server")@mcp.tool()async def execute_python(code: str) -> str:    """Colab 上で Python コードを実行"""    from IPython import get_ipython    import io, sys    old_stdout = sys.stdout    sys.stdout = io.StringIO()    try:        shell = get_ipython()        result = shell.run_cell(code)        output = sys.stdout.getvalue()        return output or "Executed successfully"    finally:        sys.stdout = old_stdout@mcp.tool()async def get_gpu_info() -> str:    """GPU 情報を取得"""    import subprocess    try:        r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)        return r.stdout    except Exception as e:        return str(e)# SSE トランスポートsse = SseServerTransport("/message")@app.get("/sse")async def sse_endpoint(request: Request):    async with sse.connect_sse(        request.scope, request.receive, request._send    ) as (read_stream, write_stream):        await mcp._mcp_server.run(            read_stream, write_stream,            mcp._mcp_server.create_initialization_options(),        )@app.post("/message")async def message_endpoint(request: Request):    return await sse.post_message(        request.scope, request.receive, request._send    )public_url = ngrok.connect(8000)print(f"MCP SSE Endpoint: {public_url}/sse")nest_asyncio.apply()uvicorn.run(app, host="0.0.0.0", port=8000)

### 方法 B: fastapi-mcp による自動公開既存の FastAPI エンドポイントを MCP ツールとして自動的に公開する `fastapi-mcp` を利用すると、実装コストを大幅に削減できます。

#### fastapi-mcp による自動 MCP 化（簡易）

In [ ]:
!pip install fastapi uvicorn pyngrok fastapi-mcp nest-asynciofrom fastapi import FastAPIfrom fastapi_mcp import FastApiMCPfrom pyngrok import ngrokimport nest_asyncio, uvicornngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")app = FastAPI()@app.get("/predict/{text}")def predict(text: str):    return {"result": f"processed: {text}"}@app.post("/train")def train(data: dict):    return {"status": "training started"}# FastAPI エンドポイントを MCP ツールとして公開mcp = FastApiMCP(app, name="colab-api-server")mcp.mount()  # /mcp エンドポイントにマウントpublic_url = ngrok.connect(8000)print(f"API & MCP: {public_url}")nest_asyncio.apply()uvicorn.run(app, host="0.0.0.0", port=8000)

### 方法 C: REST API シンプル公開（MCP 不要）MCP プロトコルを使わず、純粋に REST API として Colab の計算資源を公開したい場合は `colab-code-executor` などのラッパーを利用するのが最も手軽です。

## 方法比較| 方法 | ランスポート | 難易度 | 最適な用途 || --- | --- | --- | --- || REST FastAPI + ngrok | HTTP | 低 | シンプルな GPU 推論 API || WS WebSocket + ngrok | WebSocket | 中 | リアルタイム進捗表示・ストリーミング || MCP MCP + SSE | SSE / Streamable HTTP | 中 | AI エージェント連携・ツール呼び出し || MCP fastapi-mcp | HTTP / SSE | 低 | 既存 API の MCP プロトコル化 || REST colab-code-executor | HTTP | 低 | Jupyter カーネル管理・コード実行 |

## 注意点とベストプラクティス

### 1. ngrok の動的 URL 問題無料プランではセッションごとにランダムな URL が発行されます。PWA 側で URL をハードコードするのではなく、QR コード表示・手動入力フォーム・動的設定 API などで対応してください。

### 2. CORS の設定PWA はブラウザ上で動作するため、Colab 側のサーバーで `Access-Control-Allow-Origin` ヘッダーを適切に設定しないとリクエストがブロックされます。開発中は `allow_origins=["*"]` でも、本番環境では特定のオリジンのみを許可してください。

### 3. 認証とセキュリティngrok で公開した URL はインターネット上の誰でもアクセス可能です。API キー認証・トークンベース認証・IP 制限などを組み合わせて、不正アクセスを防止してください。

### 4. Colab セッションの維持Colab の無料ランタイムは一定時間のアイドル状態で切断されます。以下のような keep-alive 処理を検討してください。

In [ ]:
import timefrom threading import Threaddef keep_alive():    while True:        time.sleep(60)        print("Keeping alive...")Thread(target=keep_alive, daemon=True).start()

### 5. ローカルランタイム接続（逆方向）Colab からローカルマシン側の PWA やサーバーに接続したい場合は、Colab の「ローカルランタイム接続」機能を使用します。ローカルで Jupyter サーバーを起動し、Colab UI から接続します。

## 推奨構成多くのユースケースでは「FastAPI + ngrok + CORS」の組み合わせが最も実用的です。AI エージェント的な統合が必要な場合は MCP over SSE を、リアルタイム性が求められる場合は WebSocket を選択してください。

## 参考リソース- Google Developers Blog- GitHub: googlecolab/colab-mcp- GitHub: fastapi_mcp- PyPI: colab-code-executor- Colab Local Runtimes- GitHub Gist: colab_websocket_proxy.py